# Defensive-line get-off submission

This notebook recomputes the 2023–24 study in `writeup/getoff.md`. It does not run the closed receiver-cut notebook, and it does not read 2025 get-off.

## Paths

- Local data: `nfl-big-data-bowl-2027/` next to this repository, or `BDB_DATA_ROOT`.
- Kaggle data: `/kaggle/input/nfl-big-data-bowl-2027`.
- Package: the next cell inserts `src` on `sys.path`. On Kaggle, attach this repository as a dataset and set `BDB_CODE_ROOT` to that dataset's directory (the folder that contains `src` and `config`). This local run does not execute on Kaggle.


In [ ]:
from pathlib import Path
import os
import sys

CODE = Path(os.environ["BDB_CODE_ROOT"]).resolve() if os.environ.get("BDB_CODE_ROOT") else Path.cwd().resolve()
if not (CODE / "config" / "getoff_v1.yaml").exists():
    for parent in [CODE, *CODE.parents]:
        if (parent / "config" / "getoff_v1.yaml").exists():
            CODE = parent
            break
os.chdir(CODE)
os.environ["BDB2027_ROOT"] = str(CODE)
src = str(CODE / "src")
sys.path.insert(0, src)
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")

candidates = [
    Path(os.environ["BDB_DATA_ROOT"]) if os.environ.get("BDB_DATA_ROOT") else None,
    CODE / "nfl-big-data-bowl-2027",
    Path("/kaggle/input/nfl-big-data-bowl-2027"),
]
data = next(path for path in candidates if path is not None and (path / "players.csv").exists())
os.environ["BDB_DATA_ROOT"] = str(data)
print("code", CODE)
print("data", data)


In [ ]:
from bdb2027.getoff import common_penalty_diagnostic, run
import pandas as pd

summary = run(CODE)
assert summary["n_2023"] == 18 and summary["n_2024"] == 19
assert summary["chronological_mae"]["M1"] == summary["chronological_mae"]["M1"]
diagnostic = common_penalty_diagnostic(pd.read_csv(CODE / "outputs/getoff_v1/player_ledger.csv"))
print("tuned 2024 MAE", {k: round(v, 5) for k, v in summary["chronological_mae"].items()})
print("M2-M1", round(summary["delta_m2_minus_m1"], 5), summary["delta_bootstrap"])
print("diagnostic M1 alpha 3", diagnostic["mae_m1_alpha_3"])
print("diagnostic M2 alpha 3", diagnostic["mae_m2_alpha_3"])
assert abs(diagnostic["mae_m1_alpha_3"] - 0.0594126898) < 1e-9
assert abs(diagnostic["mae_m2_alpha_3"] - 0.0607088708) < 1e-9
print("2025 was not used. Kaggle execution of this notebook has not been run.")


In [ ]:
from IPython.display import Image, display
for name in ["fig_repeats.png", "fig_prediction.png", "fig_translation.png"]:
    display(Image(filename=str(CODE / "outputs/getoff_v1/presentation" / name)))
